In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling
!pip install rapidfuzz

In [ ]:
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
import re
import numpy as np
import pandas as pd

In [ ]:
#lire le fichier Excel
df_Sites = pd.read_excel('/content/Liste des occupants par composants et locaux-nov-2024 (1).xlsx');

In [ ]:
#Afficher tout le df
pd.set_option('display.max_rows', None)

In [ ]:
# Utiliser la ligne d'index 5 comme en-tête
df_Sites.columns = df_Sites.iloc[5]

# Supprimer les lignes jusqu'à l'index 5 inclus
df_Sites = df_Sites.iloc[6:].reset_index(drop=True)

In [ ]:
df_Sites.columns

In [ ]:
df_Sites2 = df_Sites[
    [
        'SITE        Code G2D',
        'SITE                                              Dénomination',
        'CPT - Dénomination',
        "COMMUNE D'IMPLANTATION",
        'LCL - Dénomination'
    ]
]

In [ ]:
df_Sites2

In [ ]:
df_sites_uniques = (
    df_Sites[
        ['SITE                                              Dénomination',
         "COMMUNE D'IMPLANTATION"]
    ]
    .dropna(subset=['SITE                                              Dénomination'])
    .drop_duplicates()
    .sort_values(
        ['SITE                                              Dénomination',
         "COMMUNE D'IMPLANTATION"]
    )
    .reset_index(drop=True)
)

# Exporter vers Excel
df_sites_uniques.to_excel('sites_uniques.xlsx', index=False)

print(df_sites_uniques)

In [ ]:
df_sites_lcl = (
    df_Sites2[
        ['SITE                                              Dénomination',
         'LCL - Dénomination']
    ]
    .dropna(subset=['LCL - Dénomination'])  # enlève les lignes où le LCL est NaN
    .drop_duplicates()
    .sort_values(
        ['SITE                                              Dénomination',
         'LCL - Dénomination']
    )
    .reset_index(drop=True)
)

In [ ]:
df_sites_CPT = (
    df_Sites2[
        ['SITE                                              Dénomination',
         'CPT - Dénomination']
    ]
    .dropna(subset=['CPT - Dénomination'])  # enlève les lignes où le LCL est NaN
    .drop_duplicates()
    .sort_values(
        ['SITE                                              Dénomination',
         'CPT - Dénomination']
    )
    .reset_index(drop=True)
)

In [ ]:
df_sites_CPT

In [ ]:
# Exporter vers Excel
df_sites_CPT.to_excel('sous-sites.xlsx', index=False)

In [ ]:
# Exporter vers Excel
df_sites_lcl.to_excel('sous-sites.xlsx', index=False)

In [ ]:
#lire le fichier Excel
df_sites_lcl_2 = pd.read_excel('/content/SitesClients_dataExport.xlsx')

In [ ]:
# rename columns
df_sites_lcl = df_sites_lcl.rename(columns={
    'SITE                                              Dénomination': 'Site',
    'LCL - Dénomination': 'Sous site'
})
df_sites_lcl_2 = df_sites_lcl_2.rename(columns={
    'Agence (r_ClientsAgencesSitesClients_2)': 'Site',
    'Nom du site (NomDuSite)': 'Sous site'
})

In [ ]:
df_sites_lcl

In [ ]:
df_sites_lcl_2

In [ ]:
df_diff = (
    df_sites_lcl
    .merge(
        df_sites_lcl_2,
        on=['Site', 'Sous site'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns='_merge')
)

df_diff

In [ ]:
# Exporter vers Excel
df_diff.to_excel('sous-sites2.xlsx', index=False)

In [ ]:
#lire le fichier Excel
df_sites_lcl_3 = pd.read_excel('/content/SitesClients_dataExport (1).xlsx')

In [ ]:
df_sites_lcl_3 = df_sites_lcl_3.rename(columns={
    'Agence (r_ClientsAgencesSitesClients_2)': 'Site',
    'Nom du site (NomDuSite)': 'Sous site'
})

In [ ]:
df_diff2 = (
    df_sites_lcl
    .merge(
        df_sites_lcl_3,
        on=['Site', 'Sous site'],
        how='left',
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns='_merge')
)

df_diff2